In [ ]:
# default_exp core

# core

> This notebook has the Core functions for the Water Classification project

In [ ]:
#hide
from nbdev.showdoc import *

In [ ]:
#export
import io
import plotly
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go

import pickle

import pandas as pd

import numpy as np

from scipy.optimize import curve_fit
from sklearn import decomposition, datasets, cluster

import math

from IPython.core.display import display, HTML
import pdb

In [ ]:
show_doc(save_obj)

<h4 id="save_obj" class="doc_header"><code>save_obj</code><a href="__main__.py#L2" class="source_link" style="float:right">[source]</a></h4>

> <code>save_obj</code>(**`obj`**, **`name`**)



## Util

In [ ]:
#export
def save_obj(obj, name):
    with open(str(name), 'wb') as f:
        pickle.dump(obj, f)

def load_obj(name):
    return pickle.load(str(name))

def wavelength_range(ini_wl, last_wl, step=1):
    "Creates a range of wavelengths from initial to last, in a defined nanometers step."
    return [f'{wl}' for wl in range(ini_wl, last_wl+1, step)]
   
def calc_area(df, bands=None, col_name="area"):
    "Calc the integral of the curve and adds it to a new column."
    bands = df.columns[df.columns.str.isdigit()] if bands is None else bands
    df[col_name] = np.trapz(df.set_index('Id')[bands].to_numpy())
    return df

def normalize(df, bands=None, inplace=False):
    "Normalize the reflectance spectra by dividing all reflectance values by the area under the curve. All normalized spectra will have area=1."

    bands = df.columns[df.columns.str.isdigit()] if bands is None else bands
    calc_area(df, bands)
    df_norm = df if inplace else df.copy()
    df_norm.loc[:, bands] = df.loc[:, bands]/df.area.to_numpy()[..., None]
    return df_norm

## Charting

In [ ]:
#export
def apply_subplot(fig, subplot, position):
    for t in subplot.data:
        fig.add_trace(t, row=position[0], col=position[1])
    return fig

def showfig(fig, buttonsToRemove=[]):
    "Converts a plotly figure into a HTML graph and displays it. That is necessary to maintain the interactive functionality of plotly in the jekyll documentation, on Github."
    
    html = io.StringIO()
    buttonsToRemove += ['toggleSpikelines', 'hoverCompareCartesian', 'zoomIn2d', 
                        'zoomOut2d', 'autoScale2d', 'hoverClosestCartesian']
    pio.write_html(fig, file=html, auto_open=False,
                   config={'modeBarButtonsToRemove': buttonsToRemove,
                           'displaylogo': False,
                          }
                  )
    
    display(HTML(data=html.getvalue()))
    
    
def log_color_scatter(*args, **kwargs):
    "Proxy for px.scatter applying log to the color scale."
    
    kwargs['color'] = np.log10(kwargs['color'])
    
    fig = px.scatter(*args, **kwargs)
    
    # take care of the logarithmic colors and labels
    fig.update_traces(marker=dict(size=4))
    tickvals=np.array([0, 0.5, 1, 1.5, 2, 2.5, 3])
    ticktext=np.power(10, tickvals)
    ticktext = list(map(lambda x: round(x), ticktext.tolist()))
    fig.update_layout(coloraxis_colorbar=dict(title="SPM", tickvals=tickvals, ticktext=ticktext))
    showfig(fig)

In [ ]:
#export
def series_to_annotation(s):
    "create a formatted annotation string given the series"
    res = ""
    for name, value in s.items():
        res += f'{name}: {round(value,2)} <br>'
    
    return res

In [ ]:
#export
from collections.abc import Iterable

def cluster_and_plot_scatter(df, bands, clusters, x, y, color, continuous=True, hover_name=None, labels=None, title='cluster', log_y=False, height=300, marker_size=4):
    "Do the clustering and plot a scatter with the informed axes."
    # do the clusterring
    cl_df = clusterize(df, bands, n_clusters=clusters)
    
    if not continuous: color = cl_df[color].astype('str')
    
    # plot the scatter into a figure
    fig = px.scatter(cl_df, x=x, y=y, color=color, hover_name='Id', height=height,
                     labels=labels, log_y=log_y
                    )
    fig.update_traces(marker=dict(size=marker_size))
    
    return fig


def multi_cluster_and_plot(dfs, bands, clusters_range, x, y, color, continuous=True, hover_name=None, labels=None, title='cluster', log_y=False, log_x=False,
                           height=300, marker_size=4, dfs_names=None):
    "Creates a grid of len(dfs) rows and len(clusters_range) columns to compare different clustering results"
    
    # Convert dfs and clusters_range to list, so it does not raise expception
    dfs = [dfs] if not isinstance(dfs, list) else dfs
    clusters_range = [clusters_range] if not isinstance(clusters_range, Iterable) else clusters_range
    
    cols = len(dfs)
    rows = len(clusters_range)
    
    col_names = dfs_names if dfs_names is not None else range(len(dfs))
    
    titles = []
    for row in clusters_range:
        for col in col_names:
            titles.append(f'{col}  (clusters={row})')
            
    fig = plotly.subplots.make_subplots(rows=rows, cols=cols, subplot_titles=titles)
    
    for row, clusters in enumerate(clusters_range):
        for col, df in enumerate(dfs):
            subfig = cluster_and_plot_scatter(df, bands, clusters, x, y, color, continuous, hover_name, labels, 
                                              title=f'{clusters} clusters', 
                                              log_y=log_y, height=height, marker_size=marker_size)
            
            
            for t in subfig.data: fig.add_trace(t, row=row+1, col=col+1)
                
            fig.update_yaxes(title_text=y, type='log' if log_y else None, row=row+1, col=col+1)
            fig.update_xaxes(title_text=x, type='log' if log_x else None, row=row+1, col=col+1)
            
    fig.update_layout(showlegend=False, height=height*rows)
            
    return fig
    

## Clustering & PCA

In [ ]:
#export
def PCA_decomposition(df, calc_bands, inf_bands=None, n_components=2):
    "Applies PCA decomposition in Dataframe df, using the bands specified in calc_bands.\nInf_bands will be returned. If None is specified, the full dataframe with PCA columns will be returned"
    
    xtrain = df[calc_bands].to_numpy()
    
    pca = decomposition.PCA(n_components=n_components)
    pca_modes = pca.fit_transform(xtrain)
    

    if inf_bands is None: inf_bands = df.columns
    pca_df = pd.concat([df[inf_bands].reset_index(drop=True), 
                        pd.DataFrame(pca_modes, columns=[f'PCA{i}' for i in range(1, n_components+1)])], 
                       axis=1)
    
    return pca_df
    
    
def clusterize(df, calc_bands, inf_bands=None, n_clusters=2):

    xtrain = df[calc_bands].to_numpy()

    clustering = cluster.AgglomerativeClustering(n_clusters=n_clusters)
    clustering.fit(xtrain)
    
    if inf_bands is None: inf_bands = df.columns
    if 'cluster' in inf_bands: inf_bands = inf_bands.drop('cluster')
    
    cluster_df = pd.concat([df[inf_bands].reset_index(drop=True), 
                            pd.DataFrame(clustering.labels_, columns=['cluster'])], 
                           axis=1)
    
    return cluster_df

    

## Statistics

In [ ]:
#export
def calc_df_grouped_stats(df, groupby, variables, nameslist, funcslist):
    "Calculate the statistics of a dataframe, grouped by a field, given a list of variables and aggregate \nfunctions"
    stats = pd.DataFrame()
    
    # convert variables to a list
    variables = [variables] if isinstance(variables, str) else variables
    
    # loop through the desired statistics
    for name, func in zip(nameslist, funcslist):
        
        # create the renaming dictionary
        renaming = {var:f'{var}_{name}' for var in variables}
        
        stats = pd.concat([stats, func(df.groupby(by=groupby)[variables]).rename(columns=renaming)], axis=1)
    
    return stats

## Curve Fitting

In [ ]:
#export

# default functions to use with fit
def nir_red_ratio(x, a, b):
    return a * np.power(x, b)

def linear(x, a, b):
    return a*x+b

def expo(x, a, b):
    return a*10**x+b

def power(x, a, b):
#     pdb.set_trace()
    return a*(x)**(b)

def poly(x, a, b, c, d):
    return a*x**b+c*x+d

def nechad(red, a=610.94, c=0.2324):
    return a * red / (1 - (red / c))


In [ ]:
#export

# functions to fet a generic function into a set of data
def fit_curve(func, X, y):
    popt, pcov = curve_fit(func, X, y, check_finite=False)
    r2, rmse, SSE = calc_errors(X, y, func, popt)
    
    return popt, r2, rmse, SSE

def calc_errors(X, y, func, params=[], decimal=2):
    
    y_hat = func(X, *params)
    r2 = round(r2_score(y, y_hat), decimal)
    rmse = round(math.sqrt(mean_squared_error(y, y_hat)), decimal)
    
    SSE = round(((y-y_hat)**2).sum(), decimal)
    
    return r2, rmse, SSE

In [ ]:
#export
def plot_data_and_curve(df, X, y, color, hover_name, hover_data, xlabel, func, params=[], title="", height=600):
    
    x = X if X.ndim == 1 else X.iloc[:,0]
    
    fig = px.scatter(df[['SPM', 'Id', 'Rio/ Bacia', 'cluster']], x=x, y=y, 
                 color=color, height=height, hover_name=hover_name,
                 hover_data=hover_data, labels={'x': xlabel})
    
    if (X.ndim == 1):
        xs = np.linspace(np.min(X)*0.8, np.max(X)*1.05, 100000)
        spm = func(xs, *params)
        fig.add_trace(go.Scatter(x=xs, y=spm, mode='lines'))
    else:
        for bright in [0.02, 0.03, 0.04, 0.05]:
            xs = np.linspace(np.min(x)*0.8, np.max(x)*1.05, 100)
            xs = np.concatenate((xs[..., None], np.repeat(bright, 100, axis=0)[..., None]), axis=1)
    #         pdb.set_trace()

            spm = func(xs, *params)
            fig.add_trace(go.Scatter(x=xs[:, 0], y=spm, mode='lines'))
    
    if title == '':
        r2, rmse, SSE = calc_errors(X, y, func, params)
        title = f'Function {func.__name__}  -  R^2 = {r2}  rmse = {rmse}  SSE = {SSE}'
    
    fig.update_layout(title=title)
#     fig.update_layout(yaxis_type='log') 
    
    return fig    

def fit_and_plot_curve(df, X, y, color, hover_name, hover_data, xlabel, func, height=600):
    
    params, r2, rmse , SSE = fit_curve(func, X, y)
    
    title=f'Function {func.__name__}  -  R^2 = {r2}  rmse = {rmse}  SSE = {SSE}'

    print(title)
    
    return plot_data_and_curve(df, X, y, color, hover_name, hover_data, xlabel, func, params, title)

In [ ]:
from nbdev.export import *
notebook2script()

Converted 00_core.ipynb.
Converted 00_Data_Preparation.ipynb.
Converted 01_EDA.ipynb.
Converted 02_Unnormalized_analysis.ipynb.
Converted 03-2_Normalized_Bright_analysis.ipynb.
Converted 03_Normalized_analysis.ipynb.
Converted 04-2_SPM_Analysis-Norm_cluster.ipynb.
Converted 04_SPM_Analysis-unnorm_cluster.ipynb.
Converted 05-2_Random_Forests-Norm.ipynb.
Converted 05_Random_Forests.ipynb.
Converted 06_Clustering_assessment.ipynb.
Converted 06_Norm-Clustering_assessment.ipynb.
Converted 07-2_Clustering_assessment-norm.ipynb.
Converted 07_Clustering_assessment-Copy1.ipynb.
Converted 10-AutoEncoder.ipynb.
Converted 11-VAE.ipynb.
Converted index.ipynb.
Converted Reflectances_1.ipynb.


In [ ]:
import gdal

In [ ]:
import fastai2

In [ ]:
import nbdev

In [ ]:
fastai2.__version__

'0.0.17'

In [ ]:
import torch

In [ ]:
torch.__version__

'1.5.0'

In [ ]:
nbdev.__version__

'0.2.18'